# Gold | MOCK [WIP]

In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Grava o mock apenas se a tabela ainda não existir, para nunca sobrescrever uma dimensão real
def salvar_mock(df, table_name):
    if spark.catalog.tableExists(f"{gold_schema}.{table_name}"):
        print(f"{table_name} já existe, mock ignorado")
        return
    save_df(df, table_name)
    spark.sql(f"ALTER TABLE {gold_schema}.{table_name} SET TBLPROPERTIES ('mock' = 'true')")
    print(f"{table_name} criada (mock)")

In [0]:
# Acrescenta a chave substituta (BIGINT) como primeira coluna, numerada pela ordem informada
def com_sk(df, nome_sk, *ordem):
    return df.select(F.row_number().over(Window.orderBy(*ordem)).cast("long").alias(nome_sk), "*")

In [0]:
canais = [
    ('1', 'URA', 'Autosservico'),
    ('2', 'CHATBOT', 'Autosservico'),
    ('3', 'ATENDIMENTO INICIAL', 'Humano'),
    ('4', 'ATENDIMENTO ESPECIALIZADO', 'Humano'),
]

df_dim_canal = com_sk(
    spark.createDataFrame(canais, ["id_canal", "nome_canal", "tipo_canal"]),
    "canal_sk",
    F.col("id_canal").cast("int")
)

salvar_mock(df_dim_canal, "dim_canal")

In [0]:
motivos = [
    ('3', 'Contestação de fatura', 'FINANCEIRO', 'ALTA'),
    ('4', 'Alteração de dados cadastrais (vencimento da fatura, telefone, email)', 'CADASTRAL', 'BAIXA'),
    ('5', 'Consulta de limite', 'FINANCEIRO', 'BAIXA'),
    ('6', 'Desbloqueio de cartão', 'CARTAO', 'ALTA'),
    ('7', 'Compra não autorizada', 'CARTAO', 'ALTA'),
    ('8', 'Consulta de contrato', 'FINANCEIRO', 'MEDIA'),
    ('9', 'Problema com aplicativo', 'ATENDIMENTO', 'MEDIA'),
    ('10', 'Consulta de fatura', 'FINANCEIRO', 'BAIXA'),
    ('11', 'Bloqueio de cartão', 'CARTAO', 'ALTA'),
    ('12', 'Contratação de cartão adicional', 'CARTAO', 'MEDIA'),
    ('13', 'Dúvidas gerais sobre programa de pontos', 'BENEFICIOS', 'BAIXA'),
    ('14', 'Transferência de agência', 'CADASTRAL', 'MEDIA'),
    ('15', 'Renegociação de dívida', 'FINANCEIRO', 'ALTA'),
    ('16', 'Cancelamento de conta', 'CADASTRAL', 'ALTA'),
]

df_dim_motivo = com_sk(
    spark.createDataFrame(motivos, ["id_motivo", "nome_motivo", "categoria", "criticidade"]),
    "motivo_sk",
    F.col("id_motivo").cast("int")
)

salvar_mock(df_dim_motivo, "dim_motivo")

In [0]:
atendentes = [
    ('1', 'Marcela França', 1, '2019-04-18', '2025-11-16', 'inativo'),
    ('2', 'Zilda Esteves', 1, '2023-09-12', '2026-01-24', 'inativo'),
    ('3', 'William Marques', 1, '2019-05-28', '2025-04-15', 'inativo'),
    ('4', 'Solange Silveira', 1, '2019-10-07', None, 'ativo'),
    ('5', 'Joaquim Barros', 1, '2020-12-06', '2026-05-23', 'inativo'),
    ('6', 'Roberta Assunção', 1, '2021-10-29', None, 'ativo'),
    ('7', 'Luiza Galvão', 1, '2022-04-24', None, 'ativo'),
    ('8', 'Josefa Brito', 1, '2023-01-25', None, 'ativo'),
    ('9', 'Jaqueline Campos', 1, '2019-04-19', None, 'ativo'),
    ('10', 'Marcos Torres', 1, '2020-01-20', None, 'ativo'),
    ('11', 'Maria Eduarda Freitas', 1, '2019-01-30', None, 'ativo'),
    ('12', 'Marilene Nascimento', 1, '2021-11-25', None, 'ativo'),
    ('13', 'Daniel Fonseca', 1, '2019-06-19', None, 'ativo'),
    ('14', 'Patrícia Barbosa', 2, '2019-08-24', None, 'ativo'),
    ('15', 'Josefa Araújo', 2, '2018-12-26', None, 'ativo'),
    ('16', 'Eduardo Paiva', 2, '2022-04-28', None, 'ativo'),
    ('17', 'Roberta Garcia', 2, '2021-12-02', None, 'ativo'),
    ('18', 'Cristiane Loureiro', 2, '2019-08-27', None, 'ativo'),
    ('19', 'Daniel Siqueira', 2, '2019-12-20', None, 'ativo'),
    ('20', 'Valdir Nascimento', 2, '2019-05-22', '2025-07-05', 'inativo'),
    ('21', 'Eduardo Lacerda', 1, '2025-04-20', None, 'ativo'),
    ('22', 'Renato Prado', 2, '2025-07-15', None, 'ativo'),
    ('23', 'Solange Caldeira', 1, '2025-11-24', None, 'ativo'),
    ('24', 'Rosangela Assunção', 1, '2026-01-26', None, 'ativo'),
    ('25', 'Maria Eduarda Miranda', 1, '2026-05-30', None, 'ativo'),
]

df_dim_atendente = com_sk(
    spark.createDataFrame(
        atendentes,
        ["id_atendente", "nome_atendente", "nivel_atendimento", "data_admissao", "data_desligamento", "status_atendente"]
    )
        .withColumn("data_admissao", F.to_date("data_admissao"))
        .withColumn("data_desligamento", F.to_date("data_desligamento")),
    "atendente_sk",
    F.col("id_atendente").cast("int")
)

salvar_mock(df_dim_atendente, "dim_atendente")

In [0]:
if spark.catalog.tableExists(f"{silver_schema}.tb_clientes"):
    df_base_cliente = (
        spark.table(f"{silver_schema}.tb_clientes")
            .select("id_cliente", "regiao", "idade", "data_cadastro", "dia_vencimento_fatura")
    )
else:
    df_base_cliente = spark.table(f"{silver_schema}.tb_chamados").select("id_cliente").distinct()

df_dim_cliente = com_sk(df_base_cliente, "cliente_sk", F.col("id_cliente"))

salvar_mock(df_dim_cliente, "dim_cliente")

In [0]:
# Período coberto pelos chamados
limites = (
    spark.table(f"{silver_schema}.tb_chamados_hora")
        .agg(
            F.min(F.to_date("hora_abertura_chamado")).alias("inicio"),
            F.max(F.to_date(F.greatest("hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"))).alias("fim")
        ).first()
)

print(f"Período da dim_data: {limites['inicio']} a {limites['fim']}")

In [0]:
feriados = [
    ('2025-01-01', 'Confraternização Universal'),
    ('2025-04-18', 'Sexta-feira Santa'),
    ('2025-04-21', 'Tiradentes'),
    ('2025-05-01', 'Dia do Trabalho'),
    ('2025-09-07', 'Independência do Brasil'),
    ('2025-10-12', 'Nossa Senhora Aparecida'),
    ('2025-11-02', 'Finados'),
    ('2025-11-15', 'Proclamação da República'),
    ('2025-11-20', 'Consciência Negra'),
    ('2025-12-25', 'Natal'),
    ('2026-01-01', 'Confraternização Universal'),
    ('2026-04-03', 'Sexta-feira Santa'),
    ('2026-04-21', 'Tiradentes'),
    ('2026-05-01', 'Dia do Trabalho'),
    ('2026-09-07', 'Independência do Brasil'),
    ('2026-10-12', 'Nossa Senhora Aparecida'),
    ('2026-11-02', 'Finados'),
    ('2026-11-15', 'Proclamação da República'),
    ('2026-11-20', 'Consciência Negra'),
    ('2026-12-25', 'Natal'),
]

df_feriados = (
    spark.createDataFrame(feriados, ["data_feriado", "nome_feriado"])
        .withColumn("data_feriado", F.to_date("data_feriado"))
        .select("data_feriado")
)

In [0]:
nomes_dias = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado"]

df_dias = (
    spark.range(1)
        .select(F.explode(F.sequence(F.lit(limites["inicio"]), F.lit(limites["fim"]))).alias("dia_completo"))
        .join(df_feriados, F.col("dia_completo") == F.col("data_feriado"), "left")
        .select(
            F.date_format("dia_completo", "yyyyMMdd").cast("int").alias("data"),
            F.year("dia_completo").alias("ano"),
            F.month("dia_completo").alias("mes"),
            F.dayofmonth("dia_completo").alias("dia"),
            F.element_at(F.array(*[F.lit(d) for d in nomes_dias]), F.dayofweek("dia_completo")).alias("dia_semana"),
            F.col("data_feriado").isNotNull().alias("feriado"),
            F.dayofweek("dia_completo").isin(1, 7).alias("fim_de_semana")
        )
        .withColumn("dia_util", ~F.col("feriado") & ~F.col("fim_de_semana"))
)

df_dim_data = com_sk(
    df_dias.select("data", "ano", "mes", "dia", "dia_semana", "dia_util", "feriado", "fim_de_semana"),
    "data_sk",
    F.col("data")
)

salvar_mock(df_dim_data, "dim_data")

In [0]:
df_segundos = (
    spark.range(86400)
        .select(
            F.floor(F.col("id") / 3600).cast("int").alias("hora"),
            F.floor((F.col("id") % 3600) / 60).cast("int").alias("minuto"),
            (F.col("id") % 60).cast("int").alias("segundo")
        )
)

df_horarios = (
    df_segundos
    .withColumn("horario", F.col("hora") * 10000 + F.col("minuto") * 100 + F.col("segundo"))
    .withColumn("horario_comercial", F.col("hora").between(8, 17))
    .withColumn(
        "periodo",
        F.when(F.col("hora").between(6, 11), "manhã")
         .when(F.col("hora").between(12, 17), "tarde")
         .otherwise("noite")
    )
)

df_dim_horario = com_sk(
    df_horarios.select("horario", "hora", "minuto", "segundo", "horario_comercial", "periodo"),
    "horario_sk",
    F.col("horario")
)

salvar_mock(df_dim_horario, "dim_horario")